In [2]:

# ==========================================
# PROJETO VALIA - INTELIGÊNCIA ARTIFICIAL
# ETAPA 1 - ANALISE TRABALHISTA
# ==========================================

from datetime import datetime, timedelta

def analisar_jornada(entrada, saida, pausa_minutos):

    formato = "%H:%M"

    horario_entrada = datetime.strptime(entrada, formato)
    horario_saida = datetime.strptime(saida, formato)

    # Considera jornadas que terminam depois da meia-noite
    if horario_saida <= horario_entrada:
        horario_saida += timedelta(days=1)

    tempo_total = (horario_saida - horario_entrada).total_seconds() / 3600

    horas_pausa = pausa_minutos / 60

    horas_trabalhadas = tempo_total - horas_pausa

    resultado = {
        "entrada": entrada,
        "saida": saida,
        "pausa_minutos": pausa_minutos,
        "horas_trabalhadas": round(horas_trabalhadas, 2),
        "pontos_atencao": []
    }

    # Parâmetro inicial de análise
    if horas_trabalhadas > 8:
        resultado["pontos_atencao"].append(
            "Jornada superior a 8 horas."
        )

    if pausa_minutos < 60 and tempo_total > 6:
        resultado["pontos_atencao"].append(
            "Pausa inferior a 60 minutos em jornada superior a 6 horas."
        )

    if not resultado["pontos_atencao"]:
        resultado["pontos_atencao"].append(
            "Nenhum ponto de atenção identificado pelos parâmetros utilizados."
        )

    return resultado


# TESTE INICIAL DO VALIA

analise = analisar_jornada(
    entrada="08:00",
    saida="19:00",
    pausa_minutos=30
)

print("===== RELATORIO VALIA =====")
print("Entrada:", analise["entrada"])
print("Saída:", analise["saida"])
print("Pausa:", analise["pausa_minutos"], "minutos")
print("Horas trabalhadas:", analise["horas_trabalhadas"])

print("\nPontos de atenção:")

for ponto in analise["pontos_atencao"]:
    print("-", ponto)


===== RELATORIO VALIA =====
Entrada: 08:00
Saída: 19:00
Pausa: 30 minutos
Horas trabalhadas: 10.5

Pontos de atenção:
- Jornada superior a 8 horas.
- Pausa inferior a 60 minutos em jornada superior a 6 horas.


#Criar a base de conhecimento (RAG)

#LEITURA DOS DOCUMENTOS TRABALHISTAS

In [4]:


!pip -q install pypdf

from google.colab import files
from pypdf import PdfReader

# Selecionar os documentos PDF
arquivos = files.upload()

# Lista para armazenar os textos
documentos = []

for nome_arquivo, conteudo in arquivos.items():

    with open(nome_arquivo, "wb") as arquivo:
        arquivo.write(conteudo)

    leitor = PdfReader(nome_arquivo)

    texto_completo = ""

    for pagina in leitor.pages:
        texto_completo += pagina.extract_text() or ""

    documentos.append({
        "nome": nome_arquivo,
        "texto": texto_completo
    })

    print("\nDocumento:", nome_arquivo)
    print("Páginas:", len(leitor.pages))
    print("Caracteres extraídos:", len(texto_completo))

print("\nTotal de documentos carregados:", len(documentos))


Saving Constituicao-Compilado.pdf to Constituicao-Compilado (1).pdf
Saving Del5452compilado.pdf to Del5452compilado.pdf
Saving Lcp 150.pdf to Lcp 150.pdf

Documento: Constituicao-Compilado (1).pdf
Páginas: 165
Caracteres extraídos: 724732

Documento: Del5452compilado.pdf
Páginas: 167
Caracteres extraídos: 678931

Documento: Lcp 150.pdf
Páginas: 12
Caracteres extraídos: 41135

Total de documentos carregados: 3


#Preparar os documentos para o RAG - DIVISAO DOS DOCUMENTOS EM TRECHOS

In [5]:

tamanho_trecho = 1000
sobreposicao = 150

trechos = []

for documento in documentos:

    texto = documento["texto"]

    inicio = 0

    while inicio < len(texto):

        fim = inicio + tamanho_trecho

        trecho = texto[inicio:fim]

        if trecho.strip():
            trechos.append({
                "documento": documento["nome"],
                "conteudo": trecho
            })

        inicio += tamanho_trecho - sobreposicao

print("===== BASE DE CONHECIMENTO VALIA =====")
print("Documentos:", len(documentos))
print("Total de trechos:", len(trechos))

print("\nExemplo de trecho:")
print(trechos[0]["conteudo"][:500])


===== BASE DE CONHECIMENTO VALIA =====
Documentos: 3
Total de trechos: 1701

Exemplo de trecho:
Presidência da República
Casa Civil
Subchefia para Assuntos Jurídicos
CONSTITUIÇÃO DA REPÚBLICA FEDERATIVA DO BRASIL DE 1988
Vide Emenda
Constitucional nº 91, de
2016
Vide Emenda
Constitucional nº 106,
de 2020
Vide Emenda
Constitucional nº 107,
de 2020
(Vide Emenda
Constitucional nº 132,
de 2023)  Vigência
(Vide Emenda
Constitucional nº 132,
de 2023)  Vigência
(Vide Emenda
Constitucional nº 132,
de 2023)  Vigência
Emendas Constitucionais Emendas Constitucionais de Revisão
Ato das Disposições Con


# INSTALACAO DAS FERRAMENTAS DE BUSCA

In [7]:

!pip -q install sentence-transformers faiss-cpu


# CRIACAO DA BUSCA SEMANTICA (RAG)

In [8]:

from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# 1. Carregar modelo de embeddings
print("Carregando modelo de linguagem...")

modelo = SentenceTransformer(
    "paraphrase-multilingual-MiniLM-L12-v2"
)

# 2. Extrair textos dos documentos
textos = [trecho["conteudo"] for trecho in trechos]

# 3. Transformar os textos em embeddings
print("Transformando documentos em embeddings...")

embeddings = modelo.encode(
    textos,
    convert_to_numpy=True,
    show_progress_bar=True,
    normalize_embeddings=True
)

# 4. Criar indice de busca
dimensao = embeddings.shape[1]

indice = faiss.IndexFlatIP(dimensao)

indice.add(embeddings.astype("float32"))

print("\n===== INDICE VALIA CRIADO =====")
print("Total de documentos:", len(documentos))
print("Total de trechos indexados:", indice.ntotal)
print("Dimensao dos embeddings:", dimensao)


Carregando modelo de linguagem...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

... (7 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Transformando documentos em embeddings...


Batches:   0%|          | 0/54 [00:00<?, ?it/s]


===== INDICE VALIA CRIADO =====
Total de documentos: 3
Total de trechos indexados: 1701
Dimensao dos embeddings: 384


# TESTE DA BUSCA INTELIGENTE

In [10]:

def buscar_documentos(pergunta, quantidade=5):

    # Transformar pergunta em embedding
    vetor_pergunta = modelo.encode(
        [pergunta],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # Buscar os trechos mais relevantes
    distancias, indices = indice.search(
        vetor_pergunta.astype("float32"),
        quantidade
    )

    print("===== RESULTADOS DA BUSCA VALIA =====")
    print("\nPergunta:", pergunta)

    for posicao, indice_trecho in enumerate(indices[0]):

        trecho = trechos[indice_trecho]

        print("\n--------------------------------")
        print("Resultado:", posicao + 1)
        print("Documento:", trecho["documento"])
        print("Similaridade:", round(float(distancias[0][posicao]), 4))
        print("Conteudo:")
        print(trecho["conteudo"][:700])


# PRIMEIRA PERGUNTA
buscar_documentos(
    "Segundo o artigo 71 da CLT, qual é o intervalo intrajornada para repouso e alimentação dos trabalhadores que cumprem jornada superior a seis horas?",
    quantidade=5
)



===== RESULTADOS DA BUSCA VALIA =====

Pergunta: Segundo o artigo 71 da CLT, qual é o intervalo intrajornada para repouso e alimentação dos trabalhadores que cumprem jornada superior a seis horas?

--------------------------------
Resultado: 1
Documento: Del5452compilado.pdf
Similaridade: 0.817
Conteudo:
 das linhas de modo a ser observada a duração normal
de oito horas de trabalho.                           (Vide Decreto-Lei nº 6.361, de 1944)
§ 1º - Para o pessoal sujeito ao regime do presente artigo, depois de cada jornada de trabalho haverá um repouso
de 10 (dez) horas contínuas, no mínimo, observando-se, outrossim, o descanso semanal.
§ 2º - Para o pessoal da equipagem de trens, a que se refere o presente artigo, quando a empresa não fornecer
alimentação, em viagem, e hospedagem, no destino, concederá uma ajuda de custo para atender a tais despesas.
§ 3º - As escalas do pessoal abrangido pelo presente artigo serão organizadas de modo que não caiba a qualquer
empregado, quinzenalme

# VALIA - VERIFICACAO DO ARTIGO 71 DA CLT

In [11]:

for documento in documentos:

    if "Del5452" in documento["nome"]:

        texto = documento["texto"]

        posicao = texto.find("Art. 71")

        if posicao != -1:
            print("ARTIGO 71 ENCONTRADO!")
            print("Documento:", documento["nome"])
            print("\nConteudo encontrado:\n")
            print(texto[posicao:posicao + 2500])
        else:
            print("Artigo 71 nao encontrado no texto extraido.")


ARTIGO 71 ENCONTRADO!
Documento: Del5452compilado.pdf

Conteudo encontrado:

Art. 71 - Em qualquer trabalho contínuo, cuja duração exceda de 6 (seis) horas, é obrigatória a concessão de um
intervalo para repouso ou alimentação, o qual será, no mínimo, de 1 (uma) hora e, salvo acordo escrito ou contrato
18/11/2026, 23:26 Del5452compilado
https://www.planalto.gov.br/ccivil_03/decreto-lei/del5452compilado.htm?utm_source=chatgpt.com 11/167coletivo em contrário, não poderá exceder de 2 (duas) horas.
§ 1º - Não excedendo de 6 (seis) horas o trabalho, será, entretanto, obrigatório um intervalo de 15 (quinze)
minutos quando a duração ultrapassar 4 (quatro) horas.
§ 2º - Os intervalos de descanso não serão computados na duração do trabalho.
§ 3º O limite mínimo de uma hora para repouso ou refeição poderá ser reduzido por ato do Ministro do Trabalho,
Indústria e Comércio, quando ouvido o Serviço de Alimentação de Previdência Social, se verificar que o estabelecimento
atende integralmente às exig

# ORGANIZACAO DOS ARTIGOS

In [12]:


import re

novos_trechos = []

for documento in documentos:

    texto = documento["texto"]

    # Separar o conteúdo pelos artigos da legislação
    partes = re.split(
        r'(?m)(?=^\s*Art\.\s*\d+)',
        texto
    )

    for parte in partes:

        parte = parte.strip()

        if len(parte) > 50:

            novos_trechos.append({
                "documento": documento["nome"],
                "conteudo": parte
            })

print("===== NOVA BASE VALIA =====")
print("Documentos:", len(documentos))
print("Trechos organizados:", len(novos_trechos))

# Procurar o artigo 71 da CLT
for trecho in novos_trechos:

    if trecho["documento"].startswith("Del5452") and re.search(
        r'^\s*Art\.\s*71\b',
        trecho["conteudo"]
    ):

        print("\nARTIGO 71 ENCONTRADO!")
        print(trecho["conteudo"][:1800])


===== NOVA BASE VALIA =====
Documentos: 3
Trechos organizados: 1180

ARTIGO 71 ENCONTRADO!
Art. 71 - Em qualquer trabalho contínuo, cuja duração exceda de 6 (seis) horas, é obrigatória a concessão de um
intervalo para repouso ou alimentação, o qual será, no mínimo, de 1 (uma) hora e, salvo acordo escrito ou contrato
18/11/2026, 23:26 Del5452compilado
https://www.planalto.gov.br/ccivil_03/decreto-lei/del5452compilado.htm?utm_source=chatgpt.com 11/167coletivo em contrário, não poderá exceder de 2 (duas) horas.
§ 1º - Não excedendo de 6 (seis) horas o trabalho, será, entretanto, obrigatório um intervalo de 15 (quinze)
minutos quando a duração ultrapassar 4 (quatro) horas.
§ 2º - Os intervalos de descanso não serão computados na duração do trabalho.
§ 3º O limite mínimo de uma hora para repouso ou refeição poderá ser reduzido por ato do Ministro do Trabalho,
Indústria e Comércio, quando ouvido o Serviço de Alimentação de Previdência Social, se verificar que o estabelecimento
atende integra



# VALIA - ATUALIZACAO DO INDICE RAG



In [13]:

# Extrair os novos textos
textos_novos = [
    trecho["conteudo"] for trecho in novos_trechos
]

print("Gerando novos embeddings...")

# Transformar os textos em embeddings
novos_embeddings = modelo.encode(
    textos_novos,
    convert_to_numpy=True,
    show_progress_bar=True,
    normalize_embeddings=True
)

# Criar novo indice
novo_indice = faiss.IndexFlatIP(
    novos_embeddings.shape[1]
)

novo_indice.add(
    novos_embeddings.astype("float32")
)

# Atualizar o indice utilizado pela busca
indice = novo_indice
trechos = novos_trechos

print("\n===== INDICE ATUALIZADO =====")
print("Total de trechos:", indice.ntotal)
print("Dimensao:", novos_embeddings.shape[1])


Gerando novos embeddings...


Batches:   0%|          | 0/37 [00:00<?, ?it/s]


===== INDICE ATUALIZADO =====
Total de trechos: 1180
Dimensao: 384


# VALIA - TESTE DA NOVA BUSCA RAG


In [14]:

buscar_documentos(
    "Segundo o artigo 71 da CLT, qual intervalo deve ser concedido para repouso e alimentação em jornadas superiores a seis horas?",
    quantidade=5
)


===== RESULTADOS DA BUSCA VALIA =====

Pergunta: Segundo o artigo 71 da CLT, qual intervalo deve ser concedido para repouso e alimentação em jornadas superiores a seis horas?

--------------------------------
Resultado: 1
Documento: Del5452compilado.pdf
Similaridade: 0.7854
Conteudo:
Art. 71 - Em qualquer trabalho contínuo, cuja duração exceda de 6 (seis) horas, é obrigatória a concessão de um
intervalo para repouso ou alimentação, o qual será, no mínimo, de 1 (uma) hora e, salvo acordo escrito ou contrato
18/11/2026, 23:26 Del5452compilado
https://www.planalto.gov.br/ccivil_03/decreto-lei/del5452compilado.htm?utm_source=chatgpt.com 11/167coletivo em contrário, não poderá exceder de 2 (duas) horas.
§ 1º - Não excedendo de 6 (seis) horas o trabalho, será, entretanto, obrigatório um intervalo de 15 (quinze)
minutos quando a duração ultrapassar 4 (quatro) horas.
§ 2º - Os intervalos de descanso não serão computados na duração do trabalho.
§ 3º O limite mínimo de uma hora pa

-------------

# VALIA - INSTALACAO DA API GEMINI

In [15]:

!pip -q install -U google-genai

print("Biblioteca Gemini instalada com sucesso!")


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.2 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.6.1 which is incompatible.
Biblioteca Gemini instalada com sucesso!


# VALIA - CONEXAO COM IA GENERATIVA

In [16]:



from google import genai
from google.colab import userdata

# Recuperar a chave dos segredos do Colab
chave = userdata.get("GEMINI_API_KEY")

# Criar cliente da IA
cliente = genai.Client(api_key=chave)

print("VALIA conectado ao Gemini!")


VALIA conectado ao Gemini!


# VALIA - PRIMEIRO TESTE DE IA GENERATIVA


In [19]:

# VALIA - TESTE COM MODELO ALTERNATIVO

import time

for tentativa in range(3):

    try:
        resposta = cliente.models.generate_content(
            model="gemini-3.5-flash-lite",
            contents="""
            Você é o assistente VALIA.

            Explique em português como pode ajudar
            trabalhadores a compreender seus direitos.
            """
        )

        print("===== VALIA INTELIGENTE =====")
        print(resposta.text)
        break

    except Exception as erro:

        print("Tentativa:", tentativa + 1)
        print("Erro:", erro)

        if tentativa < 2:
            print("Aguardando antes de tentar novamente...")
            time.sleep(5 * (2 ** tentativa))
        else:
            print("Não foi possível conectar após 3 tentativas.")


===== VALIA INTELIGENTE =====
Olá! Eu sou a **VALIA**, sua assistente virtual especializada em orientação trabalhista. 

Meu principal objetivo é tornar a legislação do trabalho acessível e fácil de entender para todos os trabalhadores, garantindo que você conheça e saiba como exercer os seus direitos.

Posso ajudar você de diversas formas no seu dia a dia profissional. Veja como:

1. **Esclarecer dúvidas sobre leis trabalhistas:** Explico regras sobre CLT, férias, 13º salário, jornada de trabalho, horas extras, adicional noturno, FGTS, entre outros temas, de forma simples e direta, sem aquela linguagem jurídica difícil.
2. **Orientação em casos de demissão:** Posso ajudar você a entender quais são as suas verbas rescisórias (quanto deve receber), prazos para pagamento, regras do seguro-desemprego e como funciona o aviso prévio.
3. **Cálculos trabalhistas básicos:** Posso fazer estimativas sobre quanto você deve receber em rescisões, férias ou horas extras (lembrando que são valores re


# VALIA - INTEGRACAO RAG + IA GENERATIVA

In [20]:

def perguntar_valia(pergunta):

    # 1. Buscar informacoes nos documentos
    vetor = modelo.encode(
        [pergunta],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    distancias, indices = indice.search(
        vetor.astype("float32"),
        5
    )

    # 2. Organizar o contexto recuperado
    contexto = ""

    for numero, indice_trecho in enumerate(indices[0]):

        trecho = trechos[indice_trecho]

        contexto += f"""
        Documento: {trecho['documento']}
        Conteudo: {trecho['conteudo']}
        """

    # 3. Enviar pergunta e contexto para o Gemini
    prompt = f"""
    Você é a assistente VALIA, especializada em
    informação e orientação trabalhista.

    Responda em português simples e objetivo.

    Utilize os documentos recuperados como fonte
    principal da resposta.

    Não invente artigos ou informações.
    Se os documentos não forem suficientes,
    informe que não encontrou informação suficiente.

    Diferencie informações gerais de possíveis
    irregularidades no caso concreto.

    CONTEXTO DOS DOCUMENTOS:
    {contexto}

    PERGUNTA DO USUÁRIO:
    {pergunta}

    Apresente também o nome do documento
    utilizado como fonte.
    """

    resposta = cliente.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )

    print("===== VALIA INTELIGENTE =====")
    print("\nPergunta:", pergunta)
    print("\nResposta:")
    print(resposta.text)


# PRIMEIRO TESTE INTEGRADO

perguntar_valia(
    "Qual intervalo deve ser concedido para uma jornada superior a seis horas segundo a CLT?"
)


===== VALIA INTELIGENTE =====

Pergunta: Qual intervalo deve ser concedido para uma jornada superior a seis horas segundo a CLT?

Resposta:
Olá! Eu sou a VALIA, sua assistente de informação e orientação trabalhista.

Com base na legislação, **informações gerais:**

Para qualquer trabalho contínuo com duração superior a 6 (seis) horas, é obrigatória a concessão de um intervalo para repouso ou alimentação de, **no mínimo, 1 (uma) hora** e, salvo acordo escrito ou contrato coletivo em contrário, não poderá exceder de 2 (duas) horas.

* **Documento utilizado como fonte:** `Del5452compilado.pdf` (Art. 71 da CLT).

*Nota: Como você não informou dados sobre um caso concreto, não é possível apontar irregularidades específicas no momento.*


Saída da execução acima

===== VALIA INTELIGENTE =====

Pergunta: Qual intervalo deve ser concedido para uma jornada superior a seis horas segundo a CLT?

Resposta:
Olá! Eu sou a VALIA, sua assistente de informação e orientação trabalhista.

Com base na legislação, **informações gerais:**

Para qualquer trabalho contínuo com duração superior a 6 (seis) horas, é obrigatória a concessão de um intervalo para repouso ou alimentação de, **no mínimo, 1 (uma) hora** e, salvo acordo escrito ou contrato coletivo em contrário, não poderá exceder de 2 (duas) horas.

* **Documento utilizado como fonte:** `Del5452compilado.pdf` (Art. 71 da CLT).

*Nota: Como você não informou dados sobre um caso concreto, não é possível apontar irregularidades específicas no momento.*


# RELATORIO INTELIGENTE TRABALHISTA

In [23]:

def gerar_relatorio_valia(entrada, saida, pausa_minutos):

    # 1. Analisar jornada
    analise = analisar_jornada(
        entrada,
        saida,
        pausa_minutos
    )

    # 2. Buscar legislacao relacionada
    pergunta = (
        "Quais são as regras gerais da CLT sobre "
        "duração da jornada de trabalho e intervalo "
        "intrajornada? Considere os artigos 58 e 71."
    )

    vetor = modelo.encode(
        [pergunta],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    distancias, indices = indice.search(
        vetor.astype("float32"),
        5
    )

    contexto = ""

    for i in indices[0]:
        trecho = trechos[int(i)]

        contexto += (
            f"\nDocumento: {trecho['documento']}"
            f"\nConteudo: {trecho['conteudo'][:2500]}\n"
        )

    # 3. Gerar relatório com IA
    prompt = f"""
    Você é o assistente VALIA.

    Analise os dados de uma jornada de trabalho
    considerando as regras gerais da CLT.

    DADOS:
    Entrada: {entrada}
    Saída: {saida}
    Pausa: {pausa_minutos} minutos
    Horas efetivamente trabalhadas:
    {analise['horas_trabalhadas']}

    PONTOS DE ATENÇÃO:
    {analise['pontos_atencao']}

    DOCUMENTOS RECUPERADOS:
    {contexto}

    Gere um relatório em português com:

    1. Resumo da jornada
    2. Possíveis irregularidades
    3. Fundamentação legal
    4. Orientações gerais

    Não invente leis ou artigos.
    Use os documentos como referência.
    Não apresente conclusões jurídicas definitivas.
    Explique que podem existir exceções legais,
    acordos coletivos e regras específicas.
    """

    resposta = cliente.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )

    print("===== RELATORIO INTELIGENTE VALIA =====")
    print(resposta.text)


# TESTE DO RELATORIO

gerar_relatorio_valia(
    entrada="08:00",
    saida="19:00",
    pausa_minutos=30
)


===== RELATORIO INTELIGENTE VALIA =====
Olá! Eu sou o assistente **VALIA**. Com base nos dados fornecidos e na legislação trabalhista (CLT), preparei o seguinte relatório analítico da jornada de trabalho:

---

### 1. Resumo da Jornada
* **Horário de Entrada:** 08:00
* **Horário de Saída:** 19:00
* **Pausa (Intervalo Intrajornada):** 30 minutos
* **Horas efetivamente trabalhadas:** 10,5 horas

---

### 2. Possíveis Irregularidades
* **Jornada Superior a 8 Horas:** A duração diária do trabalho ultrapassa o limite padrão de 8 horas, o que exige a verificação da existência de regime de compensação de horários, banco de horas ou o pagamento de horas extras excedentes.
* **Intervalo Intrajornada Reduzido:** O intervalo para repouso e alimentação foi de 30 minutos para uma jornada superior a 6 horas. Pela regra geral da CLT, o intervalo mínimo seria de 1 hora, exigindo análise para verificar a existência de autorização em acordo ou convenção coletiva válida.

---

### 3. Fundamentação Legal


# VALIA - AJUSTE DA ANALISE TRABALHISTA

In [22]:

def analisar_jornada(entrada, saida, pausa_minutos):

    from datetime import datetime, timedelta

    formato = "%H:%M"

    inicio = datetime.strptime(entrada, formato)
    fim = datetime.strptime(saida, formato)

    if fim <= inicio:
        fim += timedelta(days=1)

    tempo_total = (fim - inicio).total_seconds() / 3600
    horas_pausa = pausa_minutos / 60
    horas_trabalhadas = tempo_total - horas_pausa

    pontos = []

    if horas_trabalhadas > 8:
        pontos.append(
            "Jornada superior a 8 horas: verificar "
            "horas extras, compensacao e regime aplicavel."
        )

    if tempo_total > 6 and pausa_minutos < 60:
        pontos.append(
            "Intervalo inferior a 60 minutos: verificar "
            "a existencia de acordo ou convencao coletiva "
            "valida que permita a reducao."
        )

    if not pontos:
        pontos.append(
            "Nenhum ponto de atencao identificado "
            "pelos parametros iniciais."
        )

    return {
        "entrada": entrada,
        "saida": saida,
        "pausa_minutos": pausa_minutos,
        "horas_trabalhadas": round(horas_trabalhadas, 2),
        "pontos_atencao": pontos
    }

print("Analise VALIA atualizada!")


Analise VALIA atualizada!


In [38]:
def gerar_relatorio_valia(entrada, saida, pausa_minutos):
    analise = analisar_jornada(entrada, saida, pausa_minutos)

    contexto = buscar_documentos(
        "CLT artigo 58 artigo 59 artigo 71 e artigo 611-A inciso III jornada de trabalho horas extras intervalo intrajornada",
        quantidade=10
    )

    documentos_relevantes = "\n\n".join(
        [
            f"Documento: {item['documento']}\n"
            f"Conteúdo: {item['conteudo']}"
            for item in contexto
        ]
    )

    prompt = f"""
Você é o assistente de análise trabalhista do projeto VALIA.

Elabore um relatório informativo com base nos dados e nos documentos fornecidos.

DADOS ANALISADOS:
Entrada: {entrada}
Saída: {saida}
Intervalo: {pausa_minutos} minutos
Horas trabalhadas: {analise['horas_trabalhadas']:.2f}

PONTOS DE ATENÇÃO:
{analise['pontos_atencao']}

DOCUMENTOS LEGAIS:
{documentos_relevantes}

REGRAS:
- Não conclua automaticamente que existe uma infração.
- Considere possíveis acordos de compensação e normas coletivas.
- Cite obrigatoriamente o artigo 611-A, inciso III, explicando que ACT ou CCT pode estabelecer intervalo intrajornada mínimo de 30 minutos para jornadas superiores a seis horas.
- Diferencie a regra geral do artigo 71, caput, da exceção prevista no artigo 611-A, III.
- Ao analisar o intervalo, considere o artigo 71 e o artigo 611-A, III, da CLT.
- Ao analisar a jornada, considere os artigos 58 e 59 da CLT.
- Ao citar o intervalo mínimo de uma hora, indique o caput do artigo 71, não o § 1º.
- O § 1º do artigo 71 trata do intervalo de 15 minutos para jornadas superiores a 4 e que não excedam 6 horas.
- Não invente artigos, leis ou informações.
- Informe quando os dados forem insuficientes.
- Cite os documentos e artigos utilizados.
- Explique que o relatório é informativo e não substitui orientação jurídica.

Organize o relatório em:
1. Resumo da jornada.
2. Pontos de atenção.
3. Fundamentação legal.
4. Informações que precisam ser verificadas.
5. Conclusão informativa.
"""

    resposta = cliente.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )

    return resposta.text


print("Relatório VALIA atualizado!")

Relatório VALIA atualizado!


In [26]:
def buscar_documentos(pergunta, quantidade=5):
    embedding_pergunta = modelo.encode(
        [pergunta],
        normalize_embeddings=True
    )

    distancias, indices_encontrados = indice.search(
        embedding_pergunta,
        quantidade
    )

    resultados = []

    for posicao, indice_trecho in enumerate(indices_encontrados[0]):
        if indice_trecho == -1:
            continue

        item = trechos[indice_trecho]

        resultados.append({
            "documento": item["documento"],
            "conteudo": item["conteudo"],
            "similaridade": float(distancias[0][posicao])
        })

    return resultados


print("Função de busca corrigida!")

Função de busca corrigida!


In [35]:
teste_busca = buscar_documentos(
    "CLT artigos 58, 59, 71 e 611-A sobre jornada e intervalo",
    quantidade=10
)

print("Quantidade de resultados:", len(teste_busca))

for item in teste_busca:
    print("\nDocumento:", item["documento"])
    print("Similaridade:", round(item["similaridade"], 4))
    print("Conteúdo:", item["conteudo"][:500])

Quantidade de resultados: 10

Documento: Del5452compilado.pdf
Similaridade: 0.5636
Conteúdo: Art. 235-E.  Para o transporte de passageiros, serão observados os seguintes dispositivos:                       (Redação
dada pela Lei nº 13.103, de 2015)         (Vigência)
I - é facultado o fracionamento do intervalo de condução do veículo previsto na Lei no 9.503, de 23 de setembro de
1997 - Código de Trânsito Brasileiro, em períodos de no mínimo 5 (cinco) minutos;                        (Incluído pela Lei nº
13.103, de 2015)         (Vigência)
II - será assegurado ao motorista intervalo mín

Documento: Del5452compilado.pdf
Similaridade: 0.5351
Conteúdo: Art. 59-B.  O não atendimento das exigências legais para compensação de jornada, inclusive quando estabelecida
mediante acordo tácito, não implica a repetição do pagamento das horas excedentes à jornada normal diária se não
ultrapassada a duração máxima semanal, sendo devido apenas o respectivo adicional.                    (Incluído pela L

In [36]:
def buscar_artigos_valia():
    artigos_alvo = ["Art. 58", "Art. 59", "Art. 71", "Art. 611-A"]

    resultados = []

    for item in trechos:
        if item["documento"] != "Del5452compilado.pdf":
            continue

        conteudo = item["conteudo"].strip()

        for artigo in artigos_alvo:
            if conteudo.startswith(artigo):
                resultados.append({
                    "documento": item["documento"],
                    "conteudo": conteudo,
                    "artigo": artigo
                })
                break

    return resultados


artigos_encontrados = buscar_artigos_valia()

print("Artigos encontrados:", len(artigos_encontrados))

for artigo in artigos_encontrados:
    print("\n", artigo["artigo"])
    print(artigo["conteudo"][:700])

Artigos encontrados: 37

 Art. 58
Art. 58 - A duração normal do trabalho, para os empregados em qualquer atividade privada, não excederá de 8
(oito) horas diárias, desde que não seja fixado expressamente outro limite.
§ 1o Não serão descontadas nem computadas como jornada extraordinária as variações de horário no registro
de ponto não excedentes de cinco minutos, observado o limite máximo de dez minutos diários.                        
(Parágrafo incluído pela Lei nº 10.243, de 19.6.2001)
§ 2º  O tempo despendido pelo empregado desde a sua residência até a efetiva ocupação do posto de trabalho e
para o seu retorno, caminhando ou por qualquer meio de transporte, inclusive o fornecido pelo empregador, não será
computado na jor

 Art. 58
Art. 58-A.  Considera-se trabalho em regime de tempo parcial aquele cuja duração não exceda a trinta horas
semanais, sem a possibilidade de horas suplementares semanais, ou, ainda, aquele cuja duração não exceda a vinte e
seis horas semanais, com a possib

#Filtrar os artigos

In [29]:
import re

def buscar_artigos_valia():
    artigos_alvo = ["58", "59", "71", "611-A"]
    resultados = {}

    for artigo in artigos_alvo:
        padrao = re.compile(
            rf"^\s*Art\.\s*{re.escape(artigo)}\s*[-.]",
            re.IGNORECASE
        )

        encontrados = []

        for item in trechos:
            if item["documento"] != "Del5452compilado.pdf":
                continue

            conteudo = item["conteudo"].strip()

            if padrao.match(conteudo):
                encontrados.append({
                    "documento": item["documento"],
                    "conteudo": conteudo
                })

        resultados[artigo] = encontrados

    return resultados


artigos_valia = buscar_artigos_valia()

for artigo, encontrados in artigos_valia.items():
    print(f"\nARTIGO {artigo}: {len(encontrados)} encontrado(s)")

    for item in encontrados:
        print(item["conteudo"][:600])


ARTIGO 58: 2 encontrado(s)
Art. 58 - A duração normal do trabalho, para os empregados em qualquer atividade privada, não excederá de 8
(oito) horas diárias, desde que não seja fixado expressamente outro limite.
§ 1o Não serão descontadas nem computadas como jornada extraordinária as variações de horário no registro
de ponto não excedentes de cinco minutos, observado o limite máximo de dez minutos diários.                        
(Parágrafo incluído pela Lei nº 10.243, de 19.6.2001)
§ 2º  O tempo despendido pelo empregado desde a sua residência até a efetiva ocupação do posto de trabalho e
para o seu retorno, caminhando 
Art. 58-A.  Considera-se trabalho em regime de tempo parcial aquele cuja duração não exceda a trinta horas
semanais, sem a possibilidade de horas suplementares semanais, ou, ainda, aquele cuja duração não exceda a vinte e
seis horas semanais, com a possibilidade de acréscimo de até seis horas suplementares semanais.                   (Redação
dada pela Lei nº 13.467, d

In [31]:
import re

texto_clt = next(
    item["texto"]
    for item in documentos
    if item["nome"] == "Del5452compilado.pdf"
)

def extrair_artigo(numero):
    padrao = rf"(?im)^\s*Art\.\s*{re.escape(numero)}\s*[-.]"
    correspondencias = list(re.finditer(padrao, texto_clt))

    if not correspondencias:
        return None

    inicio = correspondencias[0].start()

    proximo = re.search(
        r"(?im)^\s*Art\.\s*\d+(?:-[A-Z])?\s*[-.]",
        texto_clt[inicio + 1:]
    )

    fim = inicio + 1 + proximo.start() if proximo else len(texto_clt)

    return texto_clt[inicio:fim].strip()


for numero in ["58", "59", "71", "611-A"]:
    artigo = extrair_artigo(numero)

    print(f"\n{'='*40}")
    print(f"ARTIGO {numero}:")
    print(artigo[:1500] if artigo else "Não encontrado")


ARTIGO 58:
Art. 58 - A duração normal do trabalho, para os empregados em qualquer atividade privada, não excederá de 8
(oito) horas diárias, desde que não seja fixado expressamente outro limite.
§ 1o Não serão descontadas nem computadas como jornada extraordinária as variações de horário no registro
de ponto não excedentes de cinco minutos, observado o limite máximo de dez minutos diários.                        
(Parágrafo incluído pela Lei nº 10.243, de 19.6.2001)
§ 2º  O tempo despendido pelo empregado desde a sua residência até a efetiva ocupação do posto de trabalho e
para o seu retorno, caminhando ou por qualquer meio de transporte, inclusive o fornecido pelo empregador, não será
computado na jornada de trabalho, por não ser tempo à disposição do empregador.                   (Redação dada pela Lei
nº 13.467, de 2017)         (Vigência)
§ 3o  (Revogado pela Lei nº 13.467, de 2017)

ARTIGO 59:
Art. 59.  A duração diária do trabalho poderá ser acrescida de horas extras, em número 

#Conectar os artigos ao relatório

In [64]:
def gerar_relatorio_valia(entrada, saida, pausa_minutos):
    analise = analisar_jornada(entrada, saida, pausa_minutos)

    numeros = ["58", "59", "71", "611-A"]

    fundamentacao = []

    for numero in numeros:
        artigo = extrair_artigo(numero)

        if artigo:
            fundamentacao.append(
                f"ARTIGO {numero} DA CLT:\n{artigo}"
            )

    documentos_legais = "\n\n".join(fundamentacao)

    prompt = f"""
Você é o assistente de análise trabalhista do projeto VALIA.

Analise os dados abaixo utilizando os artigos da CLT fornecidos.

DADOS DA JORNADA:
Entrada: {entrada}
Saída: {saida}
Intervalo: {pausa_minutos} minutos
Total trabalhado: {analise['horas_trabalhadas']:.2f} horas.

PONTOS DE ATENÇÃO:
{analise['pontos_atencao']}

FUNDAMENTAÇÃO LEGAL:
{documentos_legais}

INSTRUÇÕES:
1. Explique a jornada registrada.
2. Analise os pontos de atenção.
3. Considere os artigos 58, 59, 71 e 611-A da CLT.
4. Explique que horas extras podem envolver compensação ou banco de horas.
5. Para intervalo de 30 minutos, verifique se existe instrumento coletivo válido.
6. Não afirme que existe irregularidade sem informações suficientes.
7. Não invente informações ou fundamentos legais.
8. Finalize com ressalva de que o relatório é informativo.

Organize a resposta em:
- Resumo da jornada
- Pontos de atenção
- Fundamentação legal
- Informações adicionais necessárias
- Conclusão

FORMATAÇÃO OBRIGATÓRIA:
- Retorne o relatório em Markdown válido.
- Utilize ## para os títulos das seções.
- Utilize ### apenas para subtítulos.
- Utilize **texto** para negrito, sempre fechando corretamente os marcadores.
- Utilize listas com hífen (-) para os tópicos.
- Não utilize asteriscos soltos ou duplicados.
- Não repita os marcadores de formatação.
- Preserve os artigos e fundamentos legais fornecidos.
- Não invente informações ou referências jurídicas.
- Retorne somente o relatório, sem comentários sobre a formatação.
"""

    resposta = cliente.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )

    return resposta.text


print("Relatório VALIA conectado à fundamentação legal!")

Relatório VALIA conectado à fundamentação legal!


#Teste do relatório completo

In [66]:
resultado = gerar_relatorio_valia(
    entrada="08:00",
    saida="19:00",
    pausa_minutos=30
)

print(resultado)

## Resumo da jornada

- **Entrada:** 08:00
- **Saída:** 19:00
- **Intervalo:** 30 minutos
- **Total trabalhado:** 10,50 horas

## Pontos de atenção

- **Jornada superior a 8 horas:** verificar horas extras, compensação e regime aplicável.
- **Intervalo inferior a 60 minutos:** verificar a existência de acordo ou convenção coletiva válida que permita a redução.

## Fundamentação legal

- **Artigo 58 da CLT:** Estabelece que a duração normal do trabalho não excederá 8 horas diárias, ressalvado o limite expressamente fixado de outra forma. Também disciplina as variações de horário no registro de ponto e o tempo de deslocamento.
- **Artigo 59 da CLT:** Determina que a duração diária do trabalho poderá ser acrescida de horas extras, em número não excedente de duas, mediante acordo individual, convenção coletiva ou acordo coletivo de trabalho, com acréscimo de pelo menos 50% para a hora extra. Prevê ainda a possibilidade de compensação de jornada e banco de horas.
- **Artigo 71 da CLT:** Reg

#Teste de intervalos


In [39]:
relatorio_teste = gerar_relatorio_valia(
    entrada="08:00",
    saida="19:00",
    pausa_minutos=30
)

print(relatorio_teste)

**PROJETO VALIA — ASSISTENTE DE ANÁLISE TRABALHISTA**  
**RELATÓRIO INFORMATIVO**

---

### 1. Resumo da jornada
Com base nos dados fornecidos para a análise, registra-se o seguinte panorama da jornada diária:
* **Entrada:** 08:00
* **Saída:** 19:00
* **Intervalo intrajornada (refeição e descanso):** 30 minutos
* **Horas trabalhadas:** 10,50 horas (10 horas e 30 minutos)

---

### 2. Pontos de atenção
* **Jornada superior a 8 horas diárias:** A duração do trabalho ultrapassa o limite constitucional e legal padrão, sendo necessário verificar a existência de regime de compensação válido, acordo individual ou coletivo, bem como o regime aplicável (conforme arts. 58 e 59 da CLT).
* **Intervalo intrajornada inferior a 60 minutos:** O intervalo registrado é de 30 minutos para uma jornada superior a seis horas. É necessário verificar a existência de Acordo Coletivo de Trabalho (ACT) ou Convenção Coletiva de Trabalho (CCT) que valide essa redução (conforme art. 71, *caput*, e art. 611-A, incis

#Teste de uma jornada regular

In [40]:
relatorio_regular = gerar_relatorio_valia(
    entrada="08:00",
    saida="17:00",
    pausa_minutos=60
)

print(relatorio_regular)

**PROJETO VALIA — ASSISTENTE DE ANÁLISE TRABALHISTA**  
**RELATÓRIO INFORMATIVO**

---

### 1. Resumo da jornada
Com base nos dados fornecidos para a análise, registra-se o seguinte panorama da jornada de trabalho:
* **Entrada:** 08:00
* **Saída:** 17:00
* **Intervalo:** 60 minutos (1 hora)
* **Horas trabalhadas:** 8,00 horas

---

### 2. Pontos de atenção
* Nenhum ponto de atenção identificado pelos parâmetros iniciais, conforme dados analisados e regras aplicadas.

---

### 3. Fundamentação legal
A análise da jornada e do intervalo intrajornada baseia-se nos seguintes dispositivos legais extraídos da Consolidação das Leis do Trabalho (CLT) — *Del5452compilado.pdf*:
* **Artigo 58:** Estabelece que a duração normal do trabalho para os empregados em atividades privadas não excederá de 8 (oito) horas diárias, desde que não seja fixado expressamente outro limite.
* **Artigo 59:** Trata da possibilidade de acréscimo de horas extras à duração diária do trabalho, mediante acordo individual, 

#Teste de uma jornada com horas excedentes

In [41]:
relatorio_teste_3 = gerar_relatorio_valia(
    entrada="08:00",
    saida="18:00",
    pausa_minutos=60
)

print(relatorio_teste_3)

**PROJETO VALIA — ASSISTENTE DE ANÁLISE TRABALHISTA**
**RELATÓRIO INFORMATIVO**

---

### 1. Resumo da jornada
Com base nos dados fornecidos, a jornada de trabalho registrada apresenta os seguintes parâmetros:
* **Entrada:** 08:00
* **Saída:** 18:00
* **Intervalo intrajornada:** 60 minutos (1 hora)
* **Horas trabalhadas (líquidas):** 9,00 horas

---

### 2. Pontos de atenção
* **Jornada superior a 8 horas:** A jornada diária efetiva é de 9 horas, ultrapassando o limite constitucional e legal padrão de 8 horas diárias. Conforme apontado, é necessário verificar a existência de horas extras, regimes de compensação e o regime de trabalho aplicável.

---

### 3. Fundamentação legal
* **Duração normal do trabalho:** O **artigo 58 do Decreto-Lei nº 5.452 (CLT)** estabelece que a duração normal do trabalho, para os empregados em qualquer atividade privada, não excederá de 8 (oito) horas diárias, desde que não seja fixado expressamente outro limite.
* **Prorrogação e compensação de jornada:** O

#Avaliação do sistema

In [42]:
testes_valia = [
    {
        "cenario": "Jornada com intervalo reduzido",
        "horas": 10.5,
        "intervalo": 30,
        "esperado": "Verificar jornada e intervalo"
    },
    {
        "cenario": "Jornada regular",
        "horas": 8,
        "intervalo": 60,
        "esperado": "Nenhum ponto de atenção"
    },
    {
        "cenario": "Jornada com horas excedentes",
        "horas": 9,
        "intervalo": 60,
        "esperado": "Verificar horas extras"
    }
]

for teste in testes_valia:
    print("Cenário:", teste["cenario"])
    print("Horas trabalhadas:", teste["horas"])
    print("Intervalo:", teste["intervalo"], "minutos")
    print("Resultado esperado:", teste["esperado"])
    print("-" * 50)

Cenário: Jornada com intervalo reduzido
Horas trabalhadas: 10.5
Intervalo: 30 minutos
Resultado esperado: Verificar jornada e intervalo
--------------------------------------------------
Cenário: Jornada regular
Horas trabalhadas: 8
Intervalo: 60 minutos
Resultado esperado: Nenhum ponto de atenção
--------------------------------------------------
Cenário: Jornada com horas excedentes
Horas trabalhadas: 9
Intervalo: 60 minutos
Resultado esperado: Verificar horas extras
--------------------------------------------------


#Avaliação automática

In [44]:
from datetime import datetime

testes_avaliacao = [
    {
        "cenario": "Jornada com intervalo reduzido",
        "entrada": "08:00",
        "saida": "19:00",
        "pausa": 30,
        "esperado_horas": 10.5,
        "esperado_ponto": True
    },
    {
        "cenario": "Jornada regular",
        "entrada": "08:00",
        "saida": "17:00",
        "pausa": 60,
        "esperado_horas": 8,
        "esperado_ponto": False
    },
    {
        "cenario": "Jornada com horas excedentes",
        "entrada": "08:00",
        "saida": "18:00",
        "pausa": 60,
        "esperado_horas": 9,
        "esperado_ponto": True
    }
]

aprovados = 0

for teste in testes_avaliacao:
    resultado = analisar_jornada(
        teste["entrada"],
        teste["saida"],
        teste["pausa"]
    )

    horas_corretas = (
        resultado["horas_trabalhadas"] == teste["esperado_horas"]
    )

    possui_ponto = any(
    "Nenhum ponto de atencao" not in ponto
    for ponto in resultado["pontos_atencao"]
)

    classificacao_correta = (
        possui_ponto == teste["esperado_ponto"]
    )

    aprovado = horas_corretas and classificacao_correta

    if aprovado:
        aprovados += 1

    print("\nCenário:", teste["cenario"])
    print("Horas calculadas:", resultado["horas_trabalhadas"])
    print("Pontos de atenção:", resultado["pontos_atencao"])
    print("Resultado:", "APROVADO" if aprovado else "REPROVADO")

print("\n========== RESULTADO FINAL ==========")
print(f"Testes aprovados: {aprovados}/3")
print(f"Taxa de aprovação: {aprovados / 3 * 100:.1f}%")


Cenário: Jornada com intervalo reduzido
Horas calculadas: 10.5
Pontos de atenção: ['Jornada superior a 8 horas: verificar horas extras, compensacao e regime aplicavel.', 'Intervalo inferior a 60 minutos: verificar a existencia de acordo ou convencao coletiva valida que permita a reducao.']
Resultado: APROVADO

Cenário: Jornada regular
Horas calculadas: 8.0
Pontos de atenção: ['Nenhum ponto de atencao identificado pelos parametros iniciais.']
Resultado: APROVADO

Cenário: Jornada com horas excedentes
Horas calculadas: 9.0
Pontos de atenção: ['Jornada superior a 8 horas: verificar horas extras, compensacao e regime aplicavel.']
Resultado: APROVADO

========== RESULTADO FINAL ==========
Testes aprovados: 3/3
Taxa de aprovação: 100.0%


#Criando a interface

In [45]:
!pip install -q ipywidgets

#Criando os campos da tela

In [46]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Campos de entrada
campo_entrada = widgets.Text(
    value="08:00",
    description="Entrada:",
    placeholder="HH:MM"
)

campo_saida = widgets.Text(
    value="17:00",
    description="Saída:",
    placeholder="HH:MM"
)

campo_intervalo = widgets.IntText(
    value=60,
    description="Intervalo:",
    style={"description_width": "initial"}
)

# Botão
botao_analisar = widgets.Button(
    description="Analisar jornada",
    button_style="success",
    icon="search"
)

# Área de resultado
area_resultado = widgets.Output()

# Organizar a interface
interface_valia = widgets.VBox([
    widgets.HTML("<h2>VALIA — Assistente de Análise Trabalhista</h2>"),
    widgets.HTML("<p>Informe os horários para analisar a jornada de trabalho.</p>"),
    campo_entrada,
    campo_saida,
    campo_intervalo,
    botao_analisar,
    area_resultado
])

display(interface_valia)

#Instalar as ferramentas da API

In [47]:
!pip install -q flask flask-cors

#Conexão com a inteligência artificial

In [48]:

from flask import Flask, request, jsonify
from flask_cors import CORS
from threading import Thread

app = Flask("VALIA_API")
CORS(app)

@app.route("/", methods=["GET"])
def inicio():
    return jsonify({
        "projeto": "VALIA",
        "status": "API funcionando"
    })

@app.route("/analisar", methods=["POST"])
def analisar():
    try:
        dados = request.get_json()

        entrada = dados.get("entrada")
        saida = dados.get("saida")
        pausa = int(dados.get("pausa_minutos"))

        if not entrada or not saida or pausa < 0:
            return jsonify({
                "erro": "Informe horários e intervalo válidos."
            }), 400

        relatorio = gerar_relatorio_valia(
            entrada=entrada,
            saida=saida,
            pausa_minutos=pausa
        )

        return jsonify({
            "status": "sucesso",
            "relatorio": relatorio
        })

    except Exception as erro:
        return jsonify({
            "erro": str(erro)
        }), 500

def iniciar_valia():
    app.run(
        host="0.0.0.0",
        port=5000,
        debug=False,
        use_reloader=False
    )

Thread(target=iniciar_valia, daemon=True).start()

print("API VALIA iniciada na porta 5000!")


API VALIA iniciada na porta 5000!


In [52]:
from pyngrok import ngrok

print("Biblioteca carregada!")

Biblioteca carregada!


In [57]:
from google.colab import userdata
from pyngrok import ngrok

# Recupera o token armazenado nos Secrets
token = userdata.get("NGROK_AUTHTOKEN")

# Configura o ngrok
ngrok.set_auth_token(token)

# Cria o endereço público para a API Flask
public_url = ngrok.connect(5000).public_url

print("API VALIA disponível em:")
print(public_url)

API VALIA disponível em:
https://crabmeat-truffle-underfed.ngrok-free.dev


In [58]:
import requests

resposta = requests.get(public_url)

print("Status:", resposta.status_code)
print("Resposta:", resposta.json())

INFO:werkzeug:127.0.0.1 - - [19/Nov/2026 04:05:14] "GET / HTTP/1.1" 200 -


Status: 200
Resposta: {'projeto': 'VALIA', 'status': 'API funcionando'}


teste

In [62]:
print(gerar_relatorio_valia(
    entrada="08:00",
    saida="19:00",
    pausa_minutos=30
))

**PROJETO VALIA — ASSISTENTE DE ANÁLISE TRABALHISTA**  
**RELATÓRIO INFORMATIVO**

---

### 1. Resumo da jornada
* **Entrada:** 08:00
* **Saída:** 19:00
* **Intervalo:** 30 minutos
* **Horas trabalhadas:** 10,50 horas (10 horas e 30 minutos)

---

### 2. Pontos de atenção
* **Jornada superior a 8 horas:** A jornada registrada totaliza 10,50 horas diárias, o que excede o limite constitucional e legal geral de 8 horas diárias. É necessário verificar a existência de horas extras, regimes de compensação e o regime aplicável (*Del5452compilado.pdf - Art. 58*).
* **Intervalo intrajornada inferior a 60 minutos:** O intervalo registrado foi de 30 minutos para uma jornada superior a 6 horas. Conforme os dados analisados, é preciso verificar a existência de acordo ou convenção coletiva válida que permita essa redução.

---

### 3. Fundamentação legal
* **Duração da jornada e horas extras:** O **artigo 58 do Decreto-Lei nº 5.452/1943 (CLT)** (*Del5452compilado.pdf*) estabelece que a duração norma